# Notebook 3A: From Two Loops to a Matrix

**Learning goal:** Derive two KVL equations, arrange them as a matrix, and solve the same system numerically and symbolically.

**Why it matters:** A reproducible calculation keeps the engineering assumptions, evidence, and explanation together.

**Prior knowledge:** KCL, KVL, resistor voltage drops, signed loop traversal, and the completed Question 4 and Question 5 circuits.

> A computed answer does not prove that the original circuit model, units, wiring, or measurements were correct. Use the checks and compare with physical evidence.


## Before you start

Use the [printable two-loop-to-matrix guide](https://andrewandrade.ca/tech-edu-resources/assets/electronics/handouts/two-loop-to-matrix-bridge.pdf) beside this notebook. Keep your returned Question 4 and Question 5 work available. Use Question 4 as the worked model, then derive Question 5 without copying its released answer.

**Workflow:** arrows $\rightarrow$ two KVL equations $\rightarrow$ standard form $\rightarrow A\mathbf{x}=\mathbf{b}$ $\rightarrow$ NumPy $\rightarrow$ SymPy $\rightarrow$ checks.

> This is a guided computational bridge. The intentional debugging stages and parameter sweep in Notebook 3 are later lessons.


## Working demonstration: define the currents

Use clockwise mesh currents $I_1$ and $I_2$. Define $I_3$ downward through the shared resistor. The mesh currents oppose each other there. Write KCL first:

$$I_1-I_2-I_3=0.$$

Now isolate the shared-branch current:

$$I_3=I_1-I_2.$$

A negative value of $I_3$ would mean that the real shared-branch current is upward, opposite the chosen arrow.


## Working demonstration: derive Loop 1

Walking clockwise gives a $+10\,\mathrm{V}$ source rise, a $-3I_1$ drop, and a $-2I_3$ drop.

Write the voltage changes before substituting Ohm's law:

$$+10-V_{R1}-V_{R3}=0$$

Replace the resistor voltages with $IR$:

$$V_{R1}=I_1R_1=3I_1,\qquad V_{R3}=I_3R_3=2I_3$$

$$+10-3I_1-2I_3=0$$

Substitute $I_3=I_1-I_2$:

$$+10-3I_1-2(I_1-I_2)=0$$

Distribute $-2$ to both terms:

$$+10-3I_1-2I_1+2I_2=0$$

Combine the $I_1$ terms:

$$+10-5I_1+2I_2=0$$

Subtract $10$ from both sides:

$$-5I_1+2I_2=-10$$

Multiply the complete equation by $-1$:

$$\boxed{5I_1-2I_2=10}.$$


## Working demonstration: derive Loop 2

Walking clockwise gives a $+6\,\mathrm{V}$ source rise and a $-5I_2$ drop. The walk crosses the shared resistor against the defined $I_3$ direction, so that term is $+2I_3$.

Write the voltage changes before substituting Ohm's law:

$$+6-V_{R2}+V_{R3}=0$$

Replace the resistor voltages with $IR$:

$$V_{R2}=I_2R_2=5I_2,\qquad V_{R3}=I_3R_3=2I_3$$

$$+6-5I_2+2I_3=0$$

Substitute $I_3=I_1-I_2$:

$$+6-5I_2+2(I_1-I_2)=0$$

Distribute $+2$ to both terms:

$$+6-5I_2+2I_1-2I_2=0$$

Reorder the current terms:

$$+6+2I_1-5I_2-2I_2=0$$

Combine the $I_2$ terms:

$$+6+2I_1-7I_2=0$$

Subtract $6$ from both sides:

$$2I_1-7I_2=-6$$

Multiply the complete equation by $-1$:

$$\boxed{-2I_1+7I_2=6}.$$


## Working demonstration: read equations into a matrix

Keep the variable order $[I_1,I_2]$ in both rows.

| Equation | coefficient of $I_1$ | coefficient of $I_2$ | right side |
|:--|--:|--:|--:|
| $5I_1-2I_2=10$ | $5$ | $-2$ | $10$ |
| $-2I_1+7I_2=6$ | $-2$ | $7$ | $6$ |

$$\underbrace{\begin{bmatrix}5&-2\\-2&7\end{bmatrix}}_{A\,[\mathrm{k\Omega}]}\underbrace{\begin{bmatrix}I_1\\I_2\end{bmatrix}}_{\mathbf{x}\,[\mathrm{mA}]}=\underbrace{\begin{bmatrix}10\\6\end{bmatrix}}_{\mathbf{b}\,[\mathrm{V}]}.$$

The units work because $[\mathrm{k\Omega}][\mathrm{mA}]=[\mathrm{V}]$.

Expand the matrix product one row at a time:

$$\begin{bmatrix}5I_1+(-2)I_2\\(-2)I_1+7I_2\end{bmatrix}=\begin{bmatrix}10\\6\end{bmatrix}.$$

The top row must read back as Equation 1 and the bottom row as Equation 2 before you use a solver.


In [ ]:
import numpy as np
import sympy as sp

variable_order = ['I1', 'I2']
A_demo = np.array([[5.0, -2.0],
                   [-2.0, 7.0]])  # kOhm
b_demo = np.array([10.0, 6.0])     # V

print('Variable order:', variable_order)
print('Row 1 ->', A_demo[0, 0], 'I1 +', A_demo[0, 1], 'I2 =', b_demo[0])
print('Row 2 ->', A_demo[1, 0], 'I1 +', A_demo[1, 1], 'I2 =', b_demo[1])
assert A_demo.shape == (2, 2)
assert b_demo.shape == (2,)


## Numerical linear algebra with NumPy

`numpy.linalg.solve(A, b)` solves $A\mathbf{x}=\mathbf{b}$ without asking you to calculate the inverse. The residual $A\mathbf{x}-\mathbf{b}$ checks the two equations entered.


In [ ]:
currents_demo_mA = np.linalg.solve(A_demo, b_demo)
I1_numeric_mA, I2_numeric_mA = currents_demo_mA
I3_numeric_mA = I1_numeric_mA - I2_numeric_mA
residual_demo_V = A_demo @ currents_demo_mA - b_demo

print(f'I1 = {I1_numeric_mA:.6f} mA')
print(f'I2 = {I2_numeric_mA:.6f} mA')
print(f'I3 = {I3_numeric_mA:.6f} mA')
print('Residuals (V):', residual_demo_V)
assert np.allclose(residual_demo_V, 0.0, atol=1e-10)


## Symbolic solution with SymPy

A symbolic solver preserves exact fractions. This makes the algebra visible before decimal rounding.


In [ ]:
I1, I2 = sp.symbols('I1 I2')
equations_demo = [
    sp.Eq(5*I1 - 2*I2, 10),
    sp.Eq(-2*I1 + 7*I2, 6),
]
exact_demo = sp.solve(equations_demo, (I1, I2), dict=True)[0]
I1_exact = exact_demo[I1]
I2_exact = exact_demo[I2]
I3_exact = sp.simplify(I1_exact - I2_exact)

print('Exact solution:', exact_demo)
print('I3 exact:', I3_exact)
print('Decimal check:', [sp.N(I1_exact, 8), sp.N(I2_exact, 8), sp.N(I3_exact, 8)])
assert sp.simplify(5*I1_exact - 2*I2_exact - 10) == 0
assert sp.simplify(-2*I1_exact + 7*I2_exact - 6) == 0
assert np.allclose([float(I1_exact), float(I2_exact)], currents_demo_mA)


## Assignment overview

**Part A:** show how the completed Question 4 circuit becomes two equations, one matrix, and two matching solver results.

**Part B:** transfer the same process to Question 5 using helper questions that do not reveal its matrix or currents.

**The essential skill:** keep the shared-current definition and variable order consistent.

**The challenge:** explain every sign instead of copying the finished matrix.

**The stumbling block:** the Loop 2 shared-resistor sign changes because the loop traversal is against $I_3$.

**The investigation:** use the official NumPy or SymPy documentation when a solver object is unfamiliar.

**The verification:** use residuals, substitution, and $I_3=I_1-I_2$.


### Student explanation - replace every prompt

**Current definitions:** Explain $I_1$, $I_2$, and $I_3$.

**Loop 1 signs:** Explain why the terms are $+10$, $-3I_1$, and $-2I_3$.

**Loop 1 algebra:** Show every line from KVL to $5I_1-2I_2=10$.

**Loop 2 signs:** Explain why the shared-resistor term is $+2I_3$.

**Loop 2 algebra:** Show every line from KVL to $-2I_1+7I_2=6$.

**Variable order and units:** State them before entering the matrix.


In [ ]:
# Keep False for a clean first run. Change to True after replacing every None.
begin_student_work = False

if begin_student_work:
    student_A = np.array([[None, None],
                          [None, None]], dtype=float)  # kOhm
    student_b = np.array([None, None], dtype=float)    # V
else:
    student_A = A_demo.copy()
    student_b = b_demo.copy()
    print('Working values loaded. Replace the None values before setting the flag to True.')

print('Row 1:', student_A[0], '=', student_b[0])
print('Row 2:', student_A[1], '=', student_b[1])


In [ ]:
# Numerical solution and residual. Run after replacing every None.
student_currents_mA = np.linalg.solve(student_A, student_b)
student_residual_V = student_A @ student_currents_mA - student_b
student_I3_mA = student_currents_mA[0] - student_currents_mA[1]

print('I1, I2 (mA):', student_currents_mA)
print('I3 (mA):', student_I3_mA)
print('Residuals (V):', student_residual_V)
assert np.allclose(student_residual_V, 0.0, atol=1e-10)


In [ ]:
# Symbolic solution: replace the coefficient and right-side blanks.
student_I1, student_I2 = sp.symbols('student_I1 student_I2')
if begin_student_work:
    student_equations = [
        sp.Eq(None*student_I1 + None*student_I2, None),
        sp.Eq(None*student_I1 + None*student_I2, None),
    ]
    # student_exact = sp.solve(student_equations, (student_I1, student_I2), dict=True)[0]
    # student_I3_exact = sp.simplify(student_exact[student_I1] - student_exact[student_I2])
    # print(student_exact, 'I3 =', student_I3_exact)
else:
    print('Symbolic student scaffold is ready. Replace the None values before setting the flag to True.')


## Comparison table - replace every blank

| Quantity | Original work | Released answer | NumPy | SymPy exact | Agreement and explanation |
|:--|--:|--:|--:|--:|:--|
| $I_1$ | ___ | ___ | ___ | ___ | ___ |
| $I_2$ | ___ | ___ | ___ | ___ | ___ |
| $I_3$ | ___ | ___ | ___ | ___ | ___ |


## Part B - transfer the method to Question 5

Do not begin with the released Question 5 answer. Your job is to generate the matrix from the circuit and algebra. The symbols $a_{11}$ through $a_{22}$ and $b_1,b_2$ below are placeholders, not Question 5 values.

### B1. Current definitions

Replace each prompt with a complete answer:

1. What directions did you choose for $I_1$, $I_2$, and $I_3$?
2. Is $I_3=I_1-I_2$ or $I_3=I_2-I_1$ for your arrows? Why?
3. Which terms are rises and which are drops in Loop 1?
4. Which terms are rises and which are drops in Loop 2?

### B2. Seven algebra lines for each loop

For **each** loop, show: (1) signed KVL with resistor-voltage symbols, (2) every $V_R=IR$ substitution, (3) the shared-current substitution, (4) distribution, (5) reordering into $[I_1,I_2]$, (6) collection of like terms, and (7) moving the constant to the right.

Finish with two boxed equations in this structure:

$$\boxed{a_{11}I_1+a_{12}I_2=b_1}$$

$$\boxed{a_{21}I_1+a_{22}I_2=b_2}.$$


### B3. Question 5 coefficient table and matrix

Replace every blank from your two boxed equations.

| Equation | coefficient of $I_1$ | coefficient of $I_2$ | right side |
|:--|--:|--:|--:|
| Equation 1 | ___ | ___ | ___ |
| Equation 2 | ___ | ___ | ___ |

Use the same variable order in both rows:

$$\underbrace{\begin{bmatrix}a_{11}&a_{12}\\a_{21}&a_{22}\end{bmatrix}}_{A}\underbrace{\begin{bmatrix}I_1\\I_2\end{bmatrix}}_{\mathbf{x}}=\underbrace{\begin{bmatrix}b_1\\b_2\end{bmatrix}}_{\mathbf{b}}.$$

Replace the six placeholders with your Question 5 values. Expand the top row back into Equation 1 and the bottom row back into Equation 2. Do not run a solver until both read-back equations match your boxed equations.


In [ ]:
# Question 5: enter coefficients only after completing the algebra and row read-back.
q5_ready = False

q5_A = np.array([[None, None],
                 [None, None]], dtype=object)  # kOhm
q5_b = np.array([None, None], dtype=object)    # V

# Use +1.0 when I3 = I1 - I2; use -1.0 when I3 = I2 - I1.
q5_shared_sign = None

if q5_ready:
    q5_A = q5_A.astype(float)
    q5_b = q5_b.astype(float)
    assert q5_shared_sign in (-1.0, 1.0)
    print('Variable order: [I1, I2]')
    print('Row 1 ->', q5_A[0, 0], 'I1 +', q5_A[0, 1], 'I2 =', q5_b[0])
    print('Row 2 ->', q5_A[1, 0], 'I1 +', q5_A[1, 1], 'I2 =', q5_b[1])
else:
    print('Complete B1-B3, replace every None, then set q5_ready = True.')


In [ ]:
# Question 5: solve only after the two printed rows match the boxed equations.
if q5_ready:
    q5_currents_mA = np.linalg.solve(q5_A, q5_b)
    q5_I3_mA = q5_shared_sign * (q5_currents_mA[0] - q5_currents_mA[1])
    q5_residual_V = q5_A @ q5_currents_mA - q5_b
    print('Question 5 I1, I2 (mA):', q5_currents_mA)
    print('Question 5 I3 (mA):', q5_I3_mA)
    print('Question 5 residuals (V):', q5_residual_V)
    assert np.allclose(q5_residual_V, 0.0, atol=1e-10)
else:
    print('Solver skipped: Question 5 matrix is not ready yet.')


## Independent challenge and progressive hints

**Challenge:** read both matrix rows back as equations, then explain why a solver can produce a clean result even when the copied coefficients represent the wrong circuit.

### Hint 1 - concept clue

The shared resistor carries $I_1-I_2$ because the mesh currents oppose each other there.

### Hint 2 - documentation clue

Search `numpy.linalg.solve A shape` or `SymPy solve simultaneous equations`.

### Hint 3 - technical clue

With order $[I_1,I_2]$, the two rows are $[5,-2]$ and $[-2,7]$, with right side $[10,6]$.


## Common errors

- Jumping to a matrix before defining $I_3$.
- Using $-2I_3$ in both loops even though Loop 2 travels against $I_3$.
- Expanding $-2(I_1-I_2)$ as $-2I_1-2I_2$ instead of $-2I_1+2I_2$.
- Changing the variable order between rows.
- Mixing ohms with milliamperes or kilohms with amperes.
- Assuming a near-zero residual proves that the circuit model is correct.


## Reflection - complete in full sentences

1. Which Loop 2 sign was easiest to miss, and why?
2. How did standard form determine the matrix rows?
3. Why did SymPy show fractions while NumPy showed decimals?
4. What did the residual verify? What could it not verify?
5. What step changed your understanding of the two-loop circuit?
6. Which Question 5 helper question caught an error before the solver ran?


## Save and submit

- [ ] Current definitions and both complete derivations are visible.
- [ ] The variable order, units, $A$, and $\mathbf{b}$ are visible.
- [ ] NumPy currents and residuals are visible.
- [ ] SymPy exact fractions and $I_3$ are visible.
- [ ] The comparison table and reflections are complete.
- [ ] Question 5 includes both seven-line rearrangements and two boxed standard-form equations.
- [ ] The Question 5 coefficient table, matrix, and two row read-backs are complete.
- [ ] Question 5 was solved only after the matrix was derived and checked.
- [ ] Restart the kernel and run all required cells from top to bottom.
- [ ] Save and download `studentnumber-lastname-kirchhoff-jupyter.ipynb`.
- [ ] Reopen the downloaded notebook before submitting it.
